In [1]:
!pip install pyvis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.7 MB/s eta 0:00:00


In [15]:
import json
import pandas as pd
import networkx as nx
import numpy as np
import itertools
from pyvis.network import Network
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import ast
from collections import Counter

df_segmentado = pd.read_parquet("/content/segmentos_clasificados_NPL_Sentiment.parquet")

# ==============================================================================
# 1. CARGA Y PREPROCESAMIENTO ROBUSTO (VERSIÓN DIAGNÓSTICO)
# ==============================================================================
def cargar_y_preparar(df_input):
    print("1. 📂 Procesando datos con Analizador Universal...")
    df = df_input.copy()

    # --- FUNCIÓN DE LIMPIEZA BLINDADA ---
    def validar_lista_universal(x):
        # CASO 1: Si es None
        if x is None:
            return []

        # CASO 2: Es un Array de Numpy (común en pandas/parquet)
        if isinstance(x, (np.ndarray, np.generic)):
            # Check if the array itself is empty or contains only NaNs (for object arrays)
            if x.size == 0 or (x.dtype == object and all(pd.isna(item) for item in x)):
                return []
            return x.tolist()

        # CASO 3: Si es un escalar NaN (después de manejar arrays)
        if pd.isna(x):
            return []

        # CASO 4: Ya es una lista de Python
        if isinstance(x, list):
            return x

        # CASO 5: Es un String (texto)
        if isinstance(x, str):
            x = x.strip()
            if not x: return []

            # [cite_start]Intento A: Evaluación literal de Python (['a', 'b']) [cite: 1]
            if x.startswith('[') and x.endswith(']'):
                try:
                    return ast.literal_eval(x)
                except:
                    pass # Si falla, probamos el siguiente

            # Intento B: Formato JSON standard (["a", "b"])
            try:
                return json.loads(x)
            except:
                pass

        # CASO 6: Fallback (retorna vacío si no entiende el formato)
        return []

    # Aplicamos la limpieza
    df['organizaciones_detectadas'] = df['organizaciones_detectadas'].apply(validar_lista_universal)

    # --- DIAGNÓSTICO RÁPIDO (Para que veas qué está pasando) ---
    ejemplos_vacios = df[df['organizaciones_detectadas'].apply(len) == 0]
    ejemplos_llenos = df[df['organizaciones_detectadas'].apply(len) > 0]

    print(f"   -> Filas totales: {len(df)}")
    print(f"   -> Filas con organizaciones detectadas: {len(ejemplos_llenos)}")

    if len(ejemplos_llenos) > 0:
        print(f"   -> Ejemplo de dato recuperado: {ejemplos_llenos['organizaciones_detectadas'].iloc[0]}")
    else:
        print("   ⚠️ CRÍTICO: No se pudo recuperar ninguna lista. Revisa el formato de tu columna origen.")
        # Si esto ocurre, imprime el valor crudo para que puedas depurar
        print(f"   -> Valor crudo (primera fila): {df_input['organizaciones_detectadas'].iloc[0]}")
        return pd.DataFrame() # Retorna vacío para detener ejecución

    # Agrupación por URL (lógica original corregida)
    # Usamos una comprensión de lista segura
    df_articulos = df.groupby('url')['organizaciones_detectadas'].apply(
        lambda x: list(set(org for lista in x for org in lista))
    ).reset_index()

    # Filtro final: Artículos con interacción (más de 1 organización)
    df_final = df_articulos[df_articulos['organizaciones_detectadas'].apply(len) > 1]

    print(f"   -> Artículos válidos con interacciones (Final): {len(df_final)}")

    return df_final

In [16]:


# ==============================================================================
# 1. CARGA Y PREPROCESAMIENTO ROBUSTO
# ==============================================================================



# ==============================================================================
# 2. CONSTRUCCIÓN DEL "ESQUELETO" (LA CURA DEL HAIRBALL)
# ==============================================================================
def construir_red_esqueleto(df, k_vecinos=3):
    """
    Construye la red manteniendo solo las 'k' conexiones más fuertes por nodo.
    k=2 genera árboles muy limpios. k=3 o 4 añade algo de densidad pero controlada.
    """
    print(f"2. 🏗️ Construyendo red (Top-{k_vecinos} Neighbors)...")

    # 1. Generar todos los pares y contar pesos (Fuerza bruta inicial)
    pares = []
    for orgs in df['organizaciones_detectadas']:
        pares.extend(list(itertools.combinations(sorted(orgs), 2)))

    conteo = Counter(pares)

    # Grafo temporal completo
    G_full = nx.Graph()
    for (u, v), peso in conteo.items():
        # Opcional: Filtro mínimo de ruido global (deben aparecer juntas al menos 2 veces)
        if peso >= 2:
            G_full.add_edge(u, v, weight=peso)

    print(f"   - Grafo Bruto: {G_full.number_of_nodes()} nodos | {G_full.number_of_edges()} bordes (Demasiado denso)")

    # 2. ALGORITMO DE PODA KNN (K-Nearest Neighbors)
    # Esta es la técnica clave mencionada en tus scripts [cite: 5, 21, 66]
    G_esqueleto = nx.Graph()

    for node in G_full.nodes():
        # Ordenamos los vecinos de este nodo por peso (de mayor a menor)
        vecinos = sorted(G_full[node].items(), key=lambda x: x[1]['weight'], reverse=True)

        # Nos quedamos estrictamente con los Top K
        top_k = vecinos[:k_vecinos]

        for vecino, datos in top_k:
            # Añadimos la arista al nuevo grafo. Al ser no dirigido, se preserva
            # si A está en el top de B, o B en el top de A (Union de conjuntos)
            if not G_esqueleto.has_edge(node, vecino):
                G_esqueleto.add_edge(node, vecino, weight=datos['weight'])

    # Eliminar nodos aislados post-poda
    G_esqueleto.remove_nodes_from(list(nx.isolates(G_esqueleto)))

    print(f"   ✅ Esqueleto Final: {G_esqueleto.number_of_nodes()} nodos | {G_esqueleto.number_of_edges()} bordes.")
    return G_esqueleto

# ==============================================================================
# 3. ESTILO Y COMUNIDADES
# ==============================================================================
def estilizar_grafo(G):
    print("3. 🎨 Aplicando detección de comunidades y estilos...")

    # Detección de Comunidades (Louvain o Greedy)
    try:
        # Intentamos usar Greedy Modularity (viene en networkx)
        communities = nx.community.greedy_modularity_communities(G, weight='weight')
        comm_map = {node: i for i, comm in enumerate(communities) for node in comm}
    except:
        comm_map = {n: 0 for n in G.nodes()}

    # Mapa de colores 'tab20' para distinguir grupos
    cmap = plt.get_cmap('tab20', max(len(set(comm_map.values())), 1)) #[cite: 8]
    grados = dict(G.degree(weight='weight'))

    for node in G.nodes():
        peso = grados.get(node, 1)
        cluster_id = comm_map.get(node, 0)

        # TAMAÑO: Logarítmico para suavizar diferencias [cite: 24, 69]
        G.nodes[node]['size'] = 10 + (np.log(peso) * 8)

        # COLOR: Asignado por comunidad
        G.nodes[node]['color'] = mcolors.to_hex(cmap(cluster_id % 20))

        # ETIQUETAS:
        # Title para Hover (información detallada)
        G.nodes[node]['title'] = f"<b>{node}</b><br>Cluster: {cluster_id}<br>Peso: {peso}"
        # Label visible (texto sobre el nodo)
        G.nodes[node]['label'] = node

        # Fuente con borde para legibilidad [cite: 71]
        G.nodes[node]['font'] = {'size': 14, 'face': 'arial', 'strokeWidth': 2, 'strokeColor': '#ffffff'}

    # ARISTAS: Grises y semitransparentes para no ensuciar la vista
    for u, v in G.edges():
        G[u][v]['color'] = "rgba(180, 180, 180, 0.3)"
        G[u][v]['width'] = 1  # Finas para reducir ruido visual

    return G

# ==============================================================================
# 4. VISUALIZACIÓN (Física ForceAtlas2Based)
# ==============================================================================
def guardar_visualizacion(G, nombre_archivo="red_optimizada.html"):
    print(f"4. 🚀 Generando HTML con física optimizada...")

    net = Network(height="900px", width="100%", bgcolor="#ffffff", font_color="black", cdn_resources='remote')
    net.from_nx(G)

    # CONFIGURACIÓN CRÍTICA PARA HAIRBALLS [cite: 74, 75, 76]
    # 'forceAtlas2Based' es superior a 'barnesHut' para separar clusters densos.
    # 'gravitationalConstant': negativo alto para repeler nodos.
    # 'springLength': bajo para mantener clusters unidos.
    net.set_options("""
    var options = {
      "nodes": {
        "shape": "dot",
        "borderWidth": 1,
        "shadow": true
      },
      "physics": {
        "enabled": true,
        "solver": "forceAtlas2Based",
        "forceAtlas2Based": {
          "theta": 0.5,
          "gravitationalConstant": -100,
          "centralGravity": 0.01,
          "springLength": 80,
          "springConstant": 0.08,
          "damping": 0.4,
          "avoidOverlap": 0.5
        },
        "stabilization": {
          "enabled": true,
          "iterations": 1000,
          "updateInterval": 25
        },
        "minVelocity": 0.75
      },
      "interaction": {
        "hover": true,
        "tooltipDelay": 200,
        "zoomView": true
      }
    }
    """)

    net.save_graph(nombre_archivo)
    print(f"   -> Archivo guardado: {nombre_archivo}")
    print("   (TIP: Abre el HTML y espera unos segundos a que la física estabilice la red)")

# ==============================================================================
# EJECUCIÓN PRINCIPAL
# ==============================================================================
if __name__ == "__main__":
    # Asumimos que 'df_segmentado' es tu DataFrame original cargado en memoria
    # Si no, usa: df_segmentado = pd.read_parquet("tu_archivo.parquet")

    if 'df_segmentado' in locals():
        # 1. Preparar
        df_clean = cargar_y_preparar(df_segmentado)

        # 2. Filtrar (El secreto contra el Hairball)
        # k=3 es un buen balance entre limpieza y detalle para Fintech
        G_final = construir_red_esqueleto(df_clean, k_vecinos=3)

        if G_final.number_of_nodes() > 0:
            # 3. Estilizar
            G_final = estilizar_grafo(G_final)

            # 4. Exportar
            guardar_visualizacion(G_final, "mapa_fintech_esqueleto.html")
        else:
            print("⚠️ La red quedó vacía tras el filtrado.")
    else:
        print("❌ Por favor carga tu DataFrame en la variable 'df_segmentado' antes de ejecutar.")

1. 📂 Procesando datos con Analizador Universal...
   -> Filas totales: 19105
   -> Filas con organizaciones detectadas: 18148
   -> Ejemplo de dato recuperado: ['Vp De Ventas']
   -> Artículos válidos con interacciones (Final): 1417
2. 🏗️ Construyendo red (Top-3 Neighbors)...
   - Grafo Bruto: 561 nodos | 1982 bordes (Demasiado denso)
   ✅ Esqueleto Final: 561 nodos | 1080 bordes.
3. 🎨 Aplicando detección de comunidades y estilos...
4. 🚀 Generando HTML con física optimizada...
   -> Archivo guardado: mapa_fintech_esqueleto.html
   (TIP: Abre el HTML y espera unos segundos a que la física estabilice la red)


In [18]:
import pandas as pd
import networkx as nx
import numpy as np
import itertools
from pyvis.network import Network
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from collections import Counter

# ==============================================================================
# 1. CARGA Y PREPARACIÓN DE DATOS
# ==============================================================================
def cargar_y_preparar(ruta_archivo):
    print(f"1. 📂 Cargando archivo: {ruta_archivo}...")
    try:
        df = pd.read_parquet(ruta_archivo)
    except Exception as e:
        print(f"❌ Error leyendo parquet: {e}")
        return None

    # Normalización defensiva: Asegurar que sean listas/tuplas reales
    # Si viene en parquet como tupla, pandas lo lee como tupla directamente.
    def normalizar_a_lista(x):
        if isinstance(x, (tuple, list, np.ndarray)):
            return list(x)
        return [] # Si es None o float(nan)

    df['organizaciones_detectadas'] = df['organizaciones_detectadas'].apply(normalizar_a_lista) # [cite: 41, 62]

    # Agrupación por URL: La unidad de relación es el artículo
    df_articulos = df.groupby('url')['organizaciones_detectadas'].apply(
        lambda x: list(set(org for lista in x for org in lista))
    ).reset_index()# [cite: 2, 42]

    # Filtramos artículos con menos de 2 entidades (no generan enlaces)
    df_final = df_articulos[df_articulos['organizaciones_detectadas'].apply(len) > 1]

    print(f"   ✅ Artículos válidos para la red: {len(df_final)}")
    return df_final

# ==============================================================================
# 2. CONSTRUCCIÓN Y CIRUGÍA DE RED (El antídoto al Hairball)
# ==============================================================================
def construir_red_optimizada(df, k_vecinos=4, peso_minimo=2):
    print("2. 🏗️ Construyendo red y aplicando poda quirúrgica...")

    # A. Generar todas las combinaciones (Aristas brutas)
    pares = []
    for orgs in df['organizaciones_detectadas']:
        if len(orgs) > 1:
            pares.extend(list(itertools.combinations(sorted(orgs), 2))) #[cite: 3, 43]

    conteo = Counter(pares)

    # B. Grafo Temporal Completo
    G_full = nx.Graph()
    for (u, v), peso in conteo.items():
        if peso >= peso_minimo: # Primer filtro: Ruido básico
            G_full.add_edge(u, v, weight=peso) #[cite: 4, 44]

    print(f"   - Grafo Bruto (filtrado por peso): {G_full.number_of_edges()} enlaces.")

    # C. ALGORITMO BACKBONE (Top-k Neighbors)
    # Para cada nodo, mantenemos SOLO sus 'k' conexiones más fuertes.
    # Esto preserva la estructura local y global pero elimina la densidad excesiva.
    print(f"   🔪 Manteniendo solo las {k_vecinos} conexiones más fuertes por nodo...")

    G_clean = nx.Graph()
    for node in G_full.nodes():
        # Ordenamos vecinos por peso (mayor a menor)
        vecinos = sorted(G_full[node].items(), key=lambda x: x[1]['weight'], reverse=True)# [cite: 5, 21]

        # Nos quedamos con los Top K
        mejores_amigos = vecinos[:k_vecinos]

        for vecino, datos in mejores_amigos:
            # Añadimos la arista al nuevo grafo (unión de conjuntos)
            if not G_clean.has_edge(node, vecino):
                G_clean.add_edge(node, vecino, weight=datos['weight']) #[cite: 6, 66]

    # Limpiamos nodos aislados resultantes
    G_clean.remove_nodes_from(list(nx.isolates(G_clean))) #[cite: 22, 67]

    print(f"   ✅ Grafo Limpio: {G_clean.number_of_nodes()} nodos | {G_clean.number_of_edges()} enlaces.")
    return G_clean

# ==============================================================================
# 3. ESTILIZACIÓN Y VISUALIZACIÓN (ForceAtlas2)
# ==============================================================================
def visualizar_red(G, nombre_salida="red_analisis_final.html"):
    print("3. 🎨 Aplicando algoritmos de comunidad y renderizado...")

    # A. Detección de Comunidades (Louvain/Greedy) para colorear
    try:
        # Usamos greedy_modularity que viene nativo en nx
        communities = nx.community.greedy_modularity_communities(G, weight='weight')
        comm_map = {node: i for i, comm in enumerate(communities) for node in comm} #[cite: 7, 68]
    except:
        comm_map = {n: 0 for n in G.nodes()}

    # Paleta de colores (tab20 es ideal para distinción categórica)
    cmap = plt.get_cmap('tab20', max(len(set(comm_map.values())), 1)) #[cite: 8, 69]
    grados = dict(G.degree(weight='weight'))

    # B. Asignación de atributos visuales
    for node in G.nodes():
        cluster_id = comm_map.get(node, 0)
        peso = grados.get(node, 1)

        # TAMAÑO: Escala logarítmica para suavizar diferencias extremas
        G.nodes[node]['size'] = 10 + (np.log(peso) * 8) #[cite: 24, 70]

        # COLOR: Basado en su comunidad
        G.nodes[node]['color'] = mcolors.to_hex(cmap(cluster_id % 20))

        # TOOLTIP: Información rica al pasar el mouse
        G.nodes[node]['title'] = f"<b>{node}</b><br>Cluster: {cluster_id}<br>Peso (Grado): {peso}"

        # ETIQUETA: Visible, con borde blanco para legibilidad
        G.nodes[node]['label'] = node
        G.nodes[node]['font'] = {'size': 16, 'strokeWidth': 3, 'strokeColor': 'white', 'color': 'black'} #[cite: 71]

    # C. Aristas discretas
    for u, v in G.edges():
        G[u][v]['color'] = "rgba(150, 150, 150, 0.2)" # Transparencia alta
        G[u][v]['width'] = 1

    # D. Configuración de PyVis con ForceAtlas2Based
    # Este solver es CRÍTICO para hairballs: fuerza la separación de clústeres.
    net = Network(height="900px", width="100%", bgcolor="#ffffff", font_color="black", cdn_resources='remote')
    net.from_nx(G)

    net.set_options("""
    var options = {
      "physics": {
        "enabled": true,
        "solver": "forceAtlas2Based",
        "forceAtlas2Based": {
          "theta": 0.5,
          "gravitationalConstant": -100,
          "centralGravity": 0.01,
          "springLength": 100,
          "springConstant": 0.08,
          "avoidOverlap": 1
        },
        "stabilization": { "iterations": 1000 }
      },
      "interaction": { "hover": true, "tooltipDelay": 200 }
    }
    """)# [cite: 74, 75, 76]

    net.save_graph(nombre_salida)
    print(f"🚀 Visualización guardada en: {nombre_salida}")

# ==============================================================================
# EJECUCIÓN
# ==============================================================================
if __name__ == "__main__":
    ARCHIVO_ENTRADA = "/content/segmentos_clasificados_NPL_Sentiment.parquet" # <--- CAMBIA ESTO

    # Flujo completo
    df = cargar_y_preparar(ARCHIVO_ENTRADA)

    if df is not None:
        # k=3 o k=4 suele ser el punto dulce entre limpieza y conectividad
        G_final = construir_red_optimizada(df, k_vecinos=4, peso_minimo=2)

        if G_final.number_of_nodes() > 0:
            visualizar_red(G_final)
        else:
            print("⚠️ La red está vacía tras el filtrado. Intenta bajar 'peso_minimo' a 1.")

1. 📂 Cargando archivo: /content/segmentos_clasificados_NPL_Sentiment.parquet...
   ✅ Artículos válidos para la red: 1417
2. 🏗️ Construyendo red y aplicando poda quirúrgica...
   - Grafo Bruto (filtrado por peso): 1982 enlaces.
   🔪 Manteniendo solo las 4 conexiones más fuertes por nodo...
   ✅ Grafo Limpio: 561 nodos | 1245 enlaces.
3. 🎨 Aplicando algoritmos de comunidad y renderizado...
🚀 Visualización guardada en: red_analisis_final.html


In [19]:
import pandas as pd
import networkx as nx
import numpy as np
import itertools
from pyvis.network import Network
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from collections import Counter
import os

# ==============================================================================
# 1. CARGA Y LIMPIEZA (Igual que antes)
# ==============================================================================
def cargar_y_preparar(ruta_archivo):
    print(f"1. 📂 Cargando archivo: {ruta_archivo}...")
    try:
        df = pd.read_parquet(ruta_archivo)
    except Exception as e:
        print(f"❌ Error leyendo parquet: {e}")
        return None

    def normalizar_a_lista(x):
        if isinstance(x, (tuple, list, np.ndarray)): return list(x)
        return []

    df['organizaciones_detectadas'] = df['organizaciones_detectadas'].apply(normalizar_a_lista)

    # Agrupar por URL
    df_articulos = df.groupby('url')['organizaciones_detectadas'].apply(
        lambda x: list(set(org for lista in x for org in lista))
    ).reset_index()

    return df_articulos[df_articulos['organizaciones_detectadas'].apply(len) > 1]

# ==============================================================================
# 2. CONSTRUCCIÓN DE LA RED GLOBAL
# ==============================================================================
def construir_red_base(df, peso_minimo=2):
    print("2. 🏗️ Construyendo red base...")
    pares = []
    for orgs in df['organizaciones_detectadas']:
        if len(orgs) > 1:
            pares.extend(list(itertools.combinations(sorted(orgs), 2)))

    conteo = Counter(pares)
    G = nx.Graph()
    for (u, v), peso in conteo.items():
        if peso >= peso_minimo:
            G.add_edge(u, v, weight=peso)

    print(f"   📊 Red Global: {G.number_of_nodes()} nodos | {G.number_of_edges()} enlaces")
    return G

# ==============================================================================
# 3. SEGMENTACIÓN Y VISUALIZACIÓN POR CLÚSTER
# ==============================================================================
def generar_clusters_vinculados(G):
    print("3. 🧩 Detectando comunidades y segmentando...")

    # A. Detección de Comunidades (Greedy Modularity)
    # Esto divide la red en grupos densamente conectados
    try:
        communities = nx.community.greedy_modularity_communities(G, weight='weight')
    except:
        print("   ⚠️ No se pudieron detectar comunidades. Verifica tu versión de NetworkX.")
        return

    # B. Filtrar clústeres relevantes (ignoramos los micro-grupos de < 10 nodos)
    clusters_grandes = [c for c in communities if len(c) > 10]
    # Ordenamos por tamaño
    clusters_grandes.sort(key=len, reverse=True)

    print(f"   -> Se encontraron {len(communities)} comunidades. Procesaremos las {len(clusters_grandes)} más grandes.")

    archivos_generados = []
    nombres_clusters = []

    # C. Generar un HTML por cada clúster
    for i, nodos in enumerate(clusters_grandes):
        # 1. Crear Subgrafo
        subG = G.subgraph(nodos).copy()

        # Identificar al nodo "Líder" (el más conectado) para nombrar el archivo
        grados = dict(subG.degree(weight='weight'))
        lider = max(grados, key=grados.get)
        nombre_clean = "".join([c if c.isalnum() else "_" for c in lider])[:20]
        filename = f"cluster_{i+1}_{nombre_clean}.html"

        titulo_legible = f"#{i+1}: {lider} ({len(nodos)} nodos)"
        nombres_clusters.append((filename, titulo_legible))

        # 2. Estilo Visual (Personalizado para el clúster)
        # Usamos Kamada-Kawai, que es excelente para desplegar redes medianas sin solapamiento
        pos = nx.kamada_kawai_layout(subG)

        for n in subG.nodes():
            subG.nodes[n]['x'] = pos[n][0] * 1500 # Coordenadas fijas para estabilidad
            subG.nodes[n]['y'] = pos[n][1] * 1500

            # Tamaño y Color
            peso = grados[n]
            subG.nodes[n]['size'] = 10 + (np.log(peso) * 10)
            subG.nodes[n]['color'] = "#97c2fc" if n != lider else "#ff7f50" # Líder en naranja
            subG.nodes[n]['label'] = n
            subG.nodes[n]['title'] = f"{n} (Conexiones: {peso})"
            subG.nodes[n]['font'] = {'size': 14, 'face': 'arial', 'background': 'white'}

        # 3. Guardar HTML con PyVis
        net = Network(height="900px", width="100%", bgcolor="#f0f0f0", font_color="black")
        net.from_nx(subG)

        # Desactivamos física porque ya calculamos posiciones con Kamada-Kawai (carga instantánea)
        net.toggle_physics(False)

        net.set_options("""
        var options = {
          "interaction": {
            "dragNodes": true,
            "zoomView": true,
            "hover": true
          }
        }
        """)

        net.save_graph(filename)
        archivos_generados.append(filename)
        print(f"   -> Generado: {filename}")

    return archivos_generados, nombres_clusters

# ==============================================================================
# 4. INYECCIÓN DEL MENÚ DE NAVEGACIÓN
# ==============================================================================
def inyectar_menu(archivos, info_clusters):
    print("4. 🔗 Vinculando archivos con menú de navegación...")

    # CSS para el menú flotante
    estilo_menu = """
    <div style='position: fixed; top: 10px; left: 10px; z-index: 9999;
                background: rgba(255, 255, 255, 0.95); padding: 15px;
                border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.15);
                font-family: sans-serif; max-height: 90vh; overflow-y: auto; border: 1px solid #ccc;'>
    <div style='margin-bottom: 10px; font-weight: bold; color: #333; border-bottom: 2px solid #007bff; padding-bottom: 5px;'>
        🔍 Explorador de Comunidades
    </div>
    """

    for archivo_actual in archivos:
        with open(archivo_actual, "r", encoding="utf-8") as f:
            contenido = f.read()

        # Generar los botones (resaltando el actual)
        botones_html = estilo_menu
        for filename, etiqueta in info_clusters:
            if filename == archivo_actual:
                style = "background: #007bff; color: white; border-left: 4px solid #004494;"
            else:
                style = "background: white; color: #555; hover: {background: #f0f0f0};"

            botones_html += f"""
            <a href='{filename}' style='text-decoration: none; display: block;
               padding: 8px 12px; margin-bottom: 4px; border-radius: 4px; font-size: 13px;
               transition: all 0.2s; {style}'>
               {etiqueta}
            </a>
            """
        botones_html += "</div>" # Cerrar div del menú

        # Insertar justo después del <body>
        nuevo_contenido = contenido.replace("<body>", f"<body>\n{botones_html}")

        with open(archivo_actual, "w", encoding="utf-8") as f:
            f.write(nuevo_contenido)

    print(f"✅ ¡Proceso completado! Abre cualquiera de los archivos (ej. {archivos[0]}) para navegar.")

# ==============================================================================
# EJECUCIÓN
# ==============================================================================
if __name__ == "__main__":
    ARCHIVO = "/content/segmentos_clasificados_NPL_Sentiment.parquet"

    # 1. Preparar
    df = cargar_y_preparar(ARCHIVO)

    if df is not None:
        # 2. Grafo Global
        G_global = construir_red_base(df, peso_minimo=2)

        # 3. Segmentar
        if G_global.number_of_nodes() > 0:
            archivos, info = generar_clusters_vinculados(G_global)

            # 4. Vincular
            if archivos:
                inyectar_menu(archivos, info)
        else:
            print("⚠️ Grafo vacío. Ajusta el peso mínimo.")

1. 📂 Cargando archivo: /content/segmentos_clasificados_NPL_Sentiment.parquet...
2. 🏗️ Construyendo red base...
   📊 Red Global: 561 nodos | 1982 enlaces
3. 🧩 Detectando comunidades y segmentando...
   -> Se encontraron 42 comunidades. Procesaremos las 6 más grandes.
   -> Generado: cluster_1_Comisión_para_el_Mer.html
   -> Generado: cluster_2_Cornershop.html
   -> Generado: cluster_3_Universidad_de_Chile.html
   -> Generado: cluster_4_Comisión_Nacional_Ba.html
   -> Generado: cluster_5_Las_Fintech.html
   -> Generado: cluster_6_Blackrock.html
4. 🔗 Vinculando archivos con menú de navegación...
✅ ¡Proceso completado! Abre cualquiera de los archivos (ej. cluster_1_Comisión_para_el_Mer.html) para navegar.


In [ ]:
def generar_clusters_estaticos(G):
    # ... (código de detección de comunidades igual al anterior) ...

    for i, nodos in enumerate(clusters_grandes):
        subG = G.subgraph(nodos).copy()

        # --- PASO CRÍTICO: CALCULAR POSICIONES FIJAS EN PYTHON ---
        # Usamos Kamada-Kawai porque es excelente para "desenredar" clústeres densos
        print(f"   Calculando layout estático para Cluster {i+1}...")

        # Scale determina qué tan "dispersos" estarán los puntos
        pos = nx.kamada_kawai_layout(subG, scale=1000)

        # Asignamos las coordenadas manualmente a los nodos de PyVis
        for n in subG.nodes():
            # NetworkX devuelve pos[n] = (x, y). PyVis necesita x, y explícitos.
            # Multiplicamos por un factor para que no queden pegados
            subG.nodes[n]['x'] = pos[n][0]
            subG.nodes[n]['y'] = pos[n][1]

            # --- APAGAR FÍSICA POR NODO ---
            subG.nodes[n]['physics'] = False

        # Crear la red
        net = Network(height="900px", width="100%", bgcolor="#f0f0f0", font_color="black")
        net.from_nx(subG)

        # --- APAGAR FÍSICA GLOBALMENTE ---
        net.toggle_physics(False)

        # Opcional: Agregar botones por si el usuario quiere activarla manualmente
        net.show_buttons(filter_=['physics'])

        filename = f"cluster_{i+1}.html"
        net.save_graph(filename)

In [24]:
import pandas as pd
import networkx as nx
import numpy as np
import itertools
from pyvis.network import Network
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from collections import Counter
import os

# ==============================================================================
# 1. CARGA Y LIMPIEZA DE DATOS
# ==============================================================================
def cargar_y_preparar(ruta_archivo):
    print(f"1. 📂 Cargando archivo: {ruta_archivo}...")
    try:
        # Intenta leer parquet. Si tu archivo es CSV, cambia a pd.read_csv
        df = pd.read_parquet(ruta_archivo)
    except Exception as e:
        print(f"❌ Error leyendo archivo: {e}")
        return None

    # Función defensiva para normalizar la columna 'organizaciones_detectadas'
    # Convierte tuplas, arrays o strings a listas de Python
    def normalizar_a_lista(x):
        if isinstance(x, (tuple, list, np.ndarray)):
            return list(x)
        # Si por alguna razón viene como string (ej. "[a, b]"), intenta evaluarlo
        if isinstance(x, str) and x.startswith('['):
            try:
                import ast
                return ast.literal_eval(x)
            except:
                return []
        return []

    df['organizaciones_detectadas'] = df['organizaciones_detectadas'].apply(normalizar_a_lista)

    # Agrupamos por URL para detectar co-ocurrencias en el mismo artículo
    df_articulos = df.groupby('url')['organizaciones_detectadas'].apply(
        lambda x: list(set(org for lista in x for org in lista))
    ).reset_index()

    # Filtramos artículos con menos de 2 entidades (no generan enlaces)
    df_final = df_articulos[df_articulos['organizaciones_detectadas'].apply(len) > 1]

    print(f"   ✅ Artículos válidos para la red: {len(df_final)}")
    return df_final

# ==============================================================================
# 2. CONSTRUCCIÓN DE LA RED GLOBAL
# ==============================================================================
def construir_red_base(df, peso_minimo=2):
    print("2. 🏗️ Construyendo red base...")
    pares = []
    for orgs in df['organizaciones_detectadas']:
        if len(orgs) > 1:
            pares.extend(list(itertools.combinations(sorted(orgs), 2)))

    conteo = Counter(pares)
    G = nx.Graph()
    for (u, v), peso in conteo.items():
        if peso >= peso_minimo:
            G.add_edge(u, v, weight=peso)

    print(f"   📊 Red Global: {G.number_of_nodes()} nodos | {G.number_of_edges()} enlaces")
    return G

# ==============================================================================
# 3. GENERACIÓN DE CLÚSTERES ESTÁTICOS (SOLUCIÓN AL JITTERING)
# ==============================================================================
def generar_clusters_estaticos(G):
    print("3. 🧩 Detectando comunidades y generando mapas estáticos...")

    # A. Detección de Comunidades (Greedy Modularity)
    try:
        communities = nx.community.greedy_modularity_communities(G, weight='weight')
    except AttributeError:
        # Fallback para versiones viejas de NetworkX
        import community # python-louvain
        partition = community.best_partition(G)
        communities = [[] for _ in range(max(partition.values()) + 1)]
        for node, comm_id in partition.items():
            communities[comm_id].append(node)

    # Filtramos clústeres pequeños (ruido)
    clusters_grandes = [c for c in communities if len(c) > 10]
    clusters_grandes.sort(key=len, reverse=True)

    print(f"   -> Procesando las {len(clusters_grandes)} comunidades principales...")

    archivos_generados = []
    info_clusters = []

    for i, nodos in enumerate(clusters_grandes):
        # 1. Crear Subgrafo
        subG = G.subgraph(nodos).copy()

        # Identificar Líder para el nombre
        grados = dict(subG.degree(weight='weight'))
        lider = max(grados, key=grados.get)

        # Nombre de archivo seguro
        nombre_clean = "".join([c if c.isalnum() else "_" for c in lider])[:20]
        filename = f"cluster_{i+1}_{nombre_clean}.html"
        label_menu = f"#{i+1}: {lider} ({len(nodos)})"

        info_clusters.append((filename, label_menu))

        # ---------------------------------------------------------
        # IMPLEMENTACIÓN OPCIÓN 1: LAYOUT PRE-CALCULADO (ESTÁTICO)
        # ---------------------------------------------------------
        print(f"      Calculando layout fijo para: {label_menu}...")

        # Calculamos coordenadas X, Y en Python usando Kamada-Kawai
        # 'scale' expande el grafo para que no quede apretado
        pos = nx.kamada_kawai_layout(subG, scale=1500)

        # Asignamos atributos visuales y coordenadas fijas
        for n in subG.nodes():
            x, y = pos[n]
            subG.nodes[n]['x'] = x
            subG.nodes[n]['y'] = y
            subG.nodes[n]['physics'] = False # ¡CLAVE! Apaga la física individual

            # Estilo
            peso = grados[n]
            subG.nodes[n]['size'] = 10 + (np.log(peso) * 8)
            subG.nodes[n]['color'] = "#97c2fc" if n != lider else "#ff7f50"
            subG.nodes[n]['label'] = n
            subG.nodes[n]['title'] = f"<b>{n}</b><br>Conexiones: {peso}"

            # Fuente legible
            subG.nodes[n]['font'] = {'size': 14, 'face': 'arial', 'background': 'white'}

        # 2. Configurar PyVis
        net = Network(height="900px", width="100%", bgcolor="#f4f4f4", font_color="black")
        net.from_nx(subG)

        # APAGADO GLOBAL DE FÍSICA
        # Esto evita que el navegador intente recalcular posiciones
        net.toggle_physics(False)



        # Opciones de interacción básicas (Zoom y Pan)
        net.set_options("""
        var options = {
          "interaction": {
            "dragNodes": true,
            "zoomView": true,
            "hover": true
          },
          "physics": {
            "enabled": false
          }
        }
        """)

        net.save_graph(filename)
        archivos_generados.append(filename)

    return archivos_generados, info_clusters

# ==============================================================================
# 4. INYECCIÓN DE MENÚ DE NAVEGACIÓN
# ==============================================================================
def inyectar_menu(archivos, info_clusters):
    print("4. 🔗 Creando sistema de navegación entre archivos...")

    estilo_menu = """
    <div style='position: fixed; top: 10px; left: 10px; z-index: 9999;
                background: rgba(255, 255, 255, 0.95); padding: 15px;
                border-radius: 8px; box-shadow: 0 4px 15px rgba(0,0,0,0.2);
                font-family: sans-serif; max-height: 90vh; overflow-y: auto;
                border: 1px solid #ccc; width: 280px;'>
    <div style='margin-bottom: 12px; font-weight: bold; color: #333;
                border-bottom: 2px solid #007bff; padding-bottom: 8px;'>
        🌐 Explorador de Comunidades
    </div>
    """

    for archivo_actual in archivos:
        with open(archivo_actual, "r", encoding="utf-8") as f:
            content = f.read()

        botones = estilo_menu
        for filename, label in info_clusters:
            # Estilo condicional para resaltar el botón activo
            if filename == archivo_actual:
                style = "background: #007bff; color: white; font-weight: bold;"
            else:
                style = "background: white; color: #555; border: 1px solid #eee;"

            botones += f"""
            <a href='{filename}' style='text-decoration: none; display: block;
               padding: 8px 10px; margin-bottom: 6px; border-radius: 4px;
               font-size: 13px; transition: all 0.2s; {style}'>
               {label}
            </a>
            """
        botones += "</div>"

        new_content = content.replace("<body>", f"<body>\n{botones}")

        with open(archivo_actual, "w", encoding="utf-8") as f:
            f.write(new_content)

    print(f"✅ ¡PROCESO FINALIZADO! Abre el archivo: {archivos[0]}")

# ==============================================================================
# EJECUCIÓN PRINCIPAL
# ==============================================================================
if __name__ == "__main__":
    # --- CONFIGURACIÓN ---
    ARCHIVO_ENTRADA = "/content/segmentos_clasificados_NPL_Sentiment.parquet"  # <--- ¡Pon el nombre de tu archivo aquí!
    # ---------------------

    # 1. Cargar
    df = cargar_y_preparar(ARCHIVO_ENTRADA)

    if df is not None:
        # 2. Grafo Global
        G_global = construir_red_base(df, peso_minimo=2)

        if G_global.number_of_nodes() > 0:
            # 3. Generar HTMLs Estáticos (Sin Jittering)
            archivos, info = generar_clusters_estaticos(G_global)

            # 4. Vincular con Menú
            if archivos:
                inyectar_menu(archivos, info)
        else:
            print("⚠️ El grafo resultante está vacío. Prueba bajando el 'peso_minimo' a 1.")

1. 📂 Cargando archivo: /content/segmentos_clasificados_NPL_Sentiment.parquet...
   ✅ Artículos válidos para la red: 1417
2. 🏗️ Construyendo red base...
   📊 Red Global: 561 nodos | 1982 enlaces
3. 🧩 Detectando comunidades y generando mapas estáticos...
   -> Procesando las 6 comunidades principales...
      Calculando layout fijo para: #1: Comisión para el Mercado Financiero (198)...
      Calculando layout fijo para: #2: Cornershop (89)...
      Calculando layout fijo para: #3: Universidad de Chile (86)...
      Calculando layout fijo para: #4: Comisión Nacional Bancaria (39)...
      Calculando layout fijo para: #5: Las Fintech (14)...
      Calculando layout fijo para: #6: Blackrock (13)...
4. 🔗 Creando sistema de navegación entre archivos...
✅ ¡PROCESO FINALIZADO! Abre el archivo: cluster_1_Comisión_para_el_Mer.html
